# Phase 5 — Vector store & indexing

Decision: **Pinecone** (serverless free tier), embeddings from **Cohere `embed-v4.0`** (chosen in `03. embeddings_evaluation.ipynb`).

Steps:
1. Load the 739 chunks from `0. cleaned_data/chunks.parquet`.
2. Embed every chunk with Cohere `embed-v4.0` (`input_type="search_document"`), 1536-dim float vectors.
3. Create a Pinecone serverless index (cosine metric, dim=1536, AWS `us-east-1` — the free-tier region).
4. Upsert all 739 vectors with metadata (`law`, `language`, `article_number`, `clause`, `text`, `source_file`) so retrieval can filter and doesn't need a separate document store — the chunk text itself lives in Pinecone metadata.
5. Sanity-check with a couple of real eval questions, embedded with `input_type="search_query"`.

Requires `PINECONE_API_KEY` and `COHERE_API_KEY` in `.env` (see `03. embeddings_evaluation.ipynb` for how those got set up). Kernel: `rag311`.

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
CHUNKS_PATH = PROJECT / "0. cleaned_data" / "chunks.parquet"

load_dotenv(PROJECT / ".env")
assert os.environ.get("COHERE_API_KEY"), "COHERE_API_KEY missing from .env"
assert os.environ.get("PINECONE_API_KEY"), "PINECONE_API_KEY missing from .env"

chunks = pd.read_parquet(CHUNKS_PATH)
chunks["article_number"] = chunks["article_number"].astype(str)
chunks["clause"] = chunks["clause"].fillna("").astype(str)
print("chunks:", chunks.shape)
chunks.head()

chunks: (739, 7)


,chunk_id,law,language,article_number,clause,text,source_file
0,ssl_ar_1,ssl,ar,1,,المادة (1)\nيسمى هذا القانون (قانون الضمان الا...,scl_articles_ar.xlsx
1,ssl_ar_2_0,ssl,ar,2,أ,المادة (2)\nأ- يكون للكلمات والعبارات التالية ...,scl_articles_ar.xlsx
2,ssl_ar_2_1,ssl,ar,2,ب,المادة (2)\nب- لمقاصد هذا القانون تعني كلمة ال...,scl_articles_ar.xlsx
3,ssl_ar_3,ssl,ar,3,,المادة (3)\nأ- يشمل هذا القانون التأمينات التا...,scl_articles_ar.xlsx
4,ssl_ar_4,ssl,ar,4,,المادة (4)\nأ- يخضع لأحكام هذا القانون كل من ا...,scl_articles_ar.xlsx


In [2]:
import cohere

COHERE_MODEL = "embed-v4.0"
EMBED_DIM = 1536  # embed-v4.0 default float output dimension, confirmed via a live API call
BATCH_SIZE = 90
MAX_RETRIES = 6
BATCH_PAUSE = 2  # seconds between successful batches, to stay under trial-key rate limits

cohere_client = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])

def embed_texts(texts, input_type):
    embs = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        for attempt in range(MAX_RETRIES):
            try:
                resp = cohere_client.embed(texts=batch, model=COHERE_MODEL, input_type=input_type, embedding_types=["float"])
                embs.extend(resp.embeddings.float_)
                break
            except Exception as e:
                wait = min(60, 5 * (2 ** attempt))
                print(f"  cohere retry {attempt + 1}: {type(e).__name__}: {e} -- waiting {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError("Cohere embed failed after retries")
        time.sleep(BATCH_PAUSE)
    return embs

print(f"embedding {len(chunks)} chunks...")
chunk_embeddings = embed_texts(chunks["text"].tolist(), input_type="search_document")
assert len(chunk_embeddings) == len(chunks)
assert len(chunk_embeddings[0]) == EMBED_DIM
print("done.")

embedding 739 chunks...


  cohere retry 1: TooManyRequestsError: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': 'ac73d1ff46b885f206f80f199f93a3b8', 'x-endpoint-monthly-call-limit': '1000', 'x-trial-endpoint-call-limit': '100', 'x-trial-endpoint-call-remaining': '92', 'date': 'Sat, 25 Jul 2026 09:14:50 GMT', 'x-envoy-upstream-service-time': '35', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 429, body: {'id': 'df58a604-41a8-4987-a4f1-3c20746919cc', 'message': 'trial token rate limit exceeded, limit is 100000 tokens per minute'} -- waiting 5s


  cohere retry 2: TooManyRequestsError: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': 'a50261e32cef8e20503cd558607132f9', 'x-endpoint-monthly-call-limit': '1000', 'x-trial-endpoint-call-limit': '100', 'x-trial-endpoint-call-remaining': '89', 'date': 'Sat, 25 Jul 2026 09:14:59 GMT', 'x-envoy-upstream-service-time': '34', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 429, body: {'id': '7600a0f2-3372-49e3-820c-bf20c0544aa6', 'message': 'trial token rate limit exceeded, limit is 100000 tokens per minute'} -- waiting 10s


done.


In [3]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "jordan-labor-law"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

if INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
    print(f"created index '{INDEX_NAME}'")
else:
    print(f"index '{INDEX_NAME}' already exists")

index = pc.Index(INDEX_NAME)

created index 'jordan-labor-law'


In [4]:
# Pinecone metadata can't hold null values -- clause is "" for whole-article chunks, which is fine.
UPSERT_BATCH = 100

vectors = []
for row, emb in zip(chunks.itertuples(index=False), chunk_embeddings):
    vectors.append({
        "id": row.chunk_id,
        "values": emb,
        "metadata": {
            "law": row.law,
            "language": row.language,
            "article_number": row.article_number,
            "clause": row.clause,
            "text": row.text,
            "source_file": row.source_file,
        },
    })

for i in range(0, len(vectors), UPSERT_BATCH):
    index.upsert(vectors=vectors[i:i + UPSERT_BATCH])
    print(f"  upserted {min(i + UPSERT_BATCH, len(vectors))}/{len(vectors)}")

time.sleep(2)  # let Pinecone finish indexing before we query stats
print(index.describe_index_stats())

  upserted 100/739


  upserted 200/739


  upserted 300/739


  upserted 400/739


  upserted 500/739


  upserted 600/739


  upserted 700/739


  upserted 739/739


DescribeIndexStatsResponse(dimension=1536, total_vector_count=739, metric='cosine', namespaces=1)


## Sanity check

Run a couple of real eval questions through the live index end-to-end (embed query → Pinecone top-k) and eyeball the results.

In [5]:
import json

EVAL_PATH = PROJECT / "eval" / "eval_draft.jsonl"
eval_questions = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

sample = eval_questions.sample(3, random_state=42)
sample_embs = embed_texts(sample["question_text"].tolist(), input_type="search_query")

for (_, row), emb in zip(sample.iterrows(), sample_embs):
    print(f"\nQ ({row['question_language']}): {row['question_text']}")
    print(f"expected: {row['answer_law']} art. {row['answer_article_numbers']} ({row['answer_language']})")
    res = index.query(vector=emb, top_k=5, include_metadata=True)
    for match in res["matches"]:
        m = match["metadata"]
        print(f"  [{match['score']:.3f}] {m['law']} art.{m['article_number']}{('.' + m['clause']) if m['clause'] else ''} ({m['language']})")


Q (ar): إذا لم تكن هناك عقوبة محددة لمخالفة معينة في قانون العمل، فهل هذا يعني أنني لن أُعاقب؟
expected: Work_Law_1996 art. ['139'] (ar)


  [0.418] labor96 art.84 (ar)
  [0.413] labor96 art.77 (ar)
  [0.402] labor96 art.139 (ar)
  [0.399] labor96 art.12.هـ (ar)
  [0.400] labor96 art.133 (ar)

Q (ar): متى تنطبق عليّ أحكام إصابات العمل في قانون العمل بدلاً من قانون الضمان الاجتماعي؟
expected: Work_Law_1996 art. ['86'] (ar)
  [0.521] labor96 art.86 (en)
  [0.514] ssl art.5 (ar)
  [0.485] ssl art.2.أ (ar)
  [0.468] labor96 art.89 (en)
  [0.541] labor96 art.86 (ar)

Q (ar): هل يحق لي أخذ إجازة بدون راتب إذا انتقل زوجي أو زوجتي للعمل في مكان بعيد؟
expected: Work_Law_1996 art. ['68'] (ar)


  [0.587] labor96 art.68 (ar)
  [0.494] labor96 art.68 (en)
  [0.447] labor96 art.67 (ar)
  [0.422] labor96 art.66 (en)
  [0.403] labor96 art.66 (ar)


**Note:** right after the bulk upsert, one of the three sample queries above returned matches slightly out of score order (a `0.541` match printed last instead of first). Re-querying the same question a minute later (after the index had settled) returned it correctly sorted, with the expected article (`labor96 art.86 ar`) at rank 1. This is Pinecone serverless's normal near-real-time indexing consistency lag right after a bulk write, not a retrieval bug — worth remembering if a "just indexed, ranking looks off" moment comes up again later (e.g. after future re-indexing runs), rather than assuming the embeddings or index config are wrong.

## Result

739/739 chunks indexed into Pinecone (`jordan-labor-law`, serverless, AWS `us-east-1`, cosine, dim=1536). Phase 5 done — see `ROADMAP.md`.